<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 05 - kNN

En este notebook se entrena un modelo base **k-Nearest Neighbors (kNN)** para predecir la variable `mature`.

kNN clasifica cada usuario según los ejemplos más cercanos en el espacio de características. Por este motivo, el modelo utiliza escalado de variables dentro de un `Pipeline`.

## Objetivo metodologico

kNN aporta un modelo de comparacion distinto a los arboles. Clasifica cada usuario segun la similitud con otros usuarios en el espacio de caracteristicas, por lo que requiere escalado de variables para que ninguna magnitud domine las distancias.


## 1. Importación de librerías y configuración inicial

Se importan las funciones necesarias para cargar datos, entrenar el modelo y guardar los resultados.

In [1]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.evaluation import evaluate_model, get_classification_report, save_metrics, save_model
from src.model_training import get_base_models, split_data
from src.visualization import save_confusion_matrix_plot

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
IMG_DIR = ROOT / "img"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carga del dataset, entrenamiento y evaluación

Se entrena el modelo kNN definido en `src/model_training.py` y se calculan las métricas principales.

### Diseno experimental

El modelo kNN se implementa como `Pipeline` con `StandardScaler`. Esta decision es necesaria porque variables como `views` tienen escalas mucho mayores que metricas como `pagerank` o `clustering`.


In [2]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
X = df.drop(columns=["new_id", "mature"])
y = df["mature"]
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
model = get_base_models(random_state=42)["knn"]
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
metrics = evaluate_model(y_test, y_pred)
metrics.update({"model": "knn", "split": "test"})
pd.DataFrame([metrics])

,accuracy,precision,recall,f1,model,split
0,0.688172,0.444444,0.264706,0.331797,knn,test


## 3. Guardado de resultados

Se guarda el informe del modelo, las métricas, el modelo entrenado y la matriz de confusión.

In [3]:
print(get_classification_report(y_test, y_pred))
save_metrics(metrics, RESULTS_DIR / "knn_base_metrics.csv")
save_model(model, MODELS_DIR / "knn_base.joblib")
save_confusion_matrix_plot(model, X_test, y_test, IMG_DIR / "knn_base_cm.png")

              precision    recall  f1-score   support

       False       0.74      0.86      0.80       658
        True       0.44      0.26      0.33       272

    accuracy                           0.69       930
   macro avg       0.59      0.56      0.56       930
weighted avg       0.65      0.69      0.66       930



## 4. Conclusiones

kNN sirve como modelo comparativo frente a árboles de decisión y Random Forest. Su rendimiento depende mucho del escalado de variables y del número de vecinos utilizado.